In [5]:
try:
    from src.baseline.vendor.dn2dem_pos import dn2dem_pos as _dn2dem_pos
except ModuleNotFoundError:
    from src.baseline.dn2dem_pos import dn2dem_pos as _dn2dem_pos


ModuleNotFoundError: No module named 'src.baseline.dn2dem_pos'

In [1]:
import numpy as np
from pathlib import Path
import os

from src.baseline.vendor.dn2dem_pos import dn2dem_pos

In [2]:
PROJECT_ROOT  = Path("../..").resolve()
NP_DIR  = PROJECT_ROOT / "data" / "np32"
FILE_LIST = list(NP_DIR.glob("*.npz"))

def load_np_stack(file_list, idx):
    """
    Load .npz file(s) from file_list and return stacked arrays.

    Parameters
    ----------
    file_list : list of Path or str
        List of file paths to .npz files.
    idx : int
        Index of file to load. 
        - If idx in [0, len(file_list)-1], return array with shape (1, ...).
        - If idx == -1, load and stack all files, shape (N, ...).
        - Otherwise, raise IndexError.

    Returns
    -------
    np.ndarray
        Loaded array(s) with an added leading axis.
    """
    n_files = len(file_list)

    if idx == -1:
        arrays = []
        for f in file_list:
            with np.load(f) as data:
                arrays.append(data["bands"])
        return np.stack(arrays, axis=0)

    elif 0 <= idx < n_files:
        with np.load(file_list[idx]) as data:
            arr = data["bands"]
        return arr[np.newaxis, ...]  # shape (1, ...)

    else:
        raise IndexError(f"Index {idx} out of range for file_list of length {n_files}")

STACK = load_np_stack(FILE_LIST, -1)

STACK.shape

(10, 6, 4096, 4096)

In [3]:
def prepare_synthetic_responses(logT_min=5.5, logT_max=7.5, n_tresp=200, nt=24, nf=6):
    logT = np.linspace(logT_min, logT_max, n_tresp)      # <- T_RESP_LOGT
    centers = np.linspace(logT_min+0.2, logT_max-0.2, nf)
    width = 0.15
    T_RESP = np.exp(-0.5*((logT[:,None]-centers[None,:])/width)**2) + 1e-30
    TEMPS = np.logspace(logT_min, logT_max, nt+1)        # DEM edges in K
    return T_RESP, logT, TEMPS

T_RESP, T_RESP_LOGT, TEMPS = prepare_synthetic_responses(n_tresp=200, nt=24, nf=6)

## As-is

Running dn2dem_pos as provided

In [5]:
# Chose Frame

i = 0

# 1) Put channels last
frame = np.moveaxis(STACK[0], 0, -1)     # (H, W, 6)

# 2) Simple error model
edn   = np.sqrt(np.clip(frame, 0, None)) + 1e-6

# 3) Sanity checks
assert frame.ndim == 3 and frame.shape[-1] == 6
assert T_RESP.shape[1] == 6                # (n_tresp, 6)
assert T_RESP.shape[0] == T_RESP_LOGT.shape[0]

# 1) Channels last and finite values only
frame = np.moveaxis(STACK[i], 0, -1).astype(np.float64, copy=False)
frame[~np.isfinite(frame)] = 0.0

# 2) Non-negative DN and positive errors (avoid zeros)
frame = np.clip(frame, 0, None)
edn = np.sqrt(frame) + 1e-6           # or your instrument error model
edn[~np.isfinite(edn)] = 1e-6
edn = np.maximum(edn, 1e-6)

# 3) Ensure T_RESP has no nonpositive entries (dn2dem_pos does a fix, but help it)
T_RESP = np.asarray(T_RESP, float)
for k in range(T_RESP.shape[1]):
    col = T_RESP[:, k]
    good = col > 0
    if good.any():
        col[~good] = col[good].min()
    else:
        # if an entire column is <=0, set to a tiny positive constant
        col[:] = 1e-30

In [6]:
# 4) Run inversion (single-threaded path will be used for small na; large na uses internal serial loop) 
demmap, edemmap, logT_bins, chisq, dn_reg = dn2dem_pos(
    frame,            # (nx, ny, nf)  == (H, W, 6)
    edn,              # same shape
    T_RESP,           # (n_tresp, 6)
    T_RESP_LOGT,      # (n_tresp,)
    TEMPS,            # (nt+1,)  edges in Kelvin
    nmu=42
)

100%|██████████| 168k/168k [06:29<00:00, 431 x10^2 DEM/s]      


## Profiling

In [7]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

try:
    %load_ext line_profiler
except Exception as e:
    print("line_profiler not available. Install with: pip install line_profiler")
    print("Continuing without detailed profiling...")

In [8]:
# Use a crop first (faster to iterate). Switch to full 4096² once happy.
frame_6hw = STACK[0, :, :1024, :1024]        # (6, H, W)
frame = np.moveaxis(frame_6hw, 0, -1).astype(np.float64, copy=False)  # (H, W, 6)
frame[~np.isfinite(frame)] = 0.0
frame = np.clip(frame, 0, None)
edn = np.sqrt(frame) + 1e-6

Actual profiling here

In [11]:
# --- 1) tiny crop so demmap_pos uses the SERIAL branch (na < 200) ---
crop_6hw = STACK[0, :, :10, :10]  # (6, 10, 10)

# Prep inputs
frame = np.moveaxis(crop_6hw, 0, -1).astype(np.float64, copy=False)  # (10,10,6)
frame[~np.isfinite(frame)] = 0.0
frame = np.clip(frame, 0, None)
edn = np.sqrt(frame) + 1e-6

# --- 2) import vanilla functions correctly ---

# --- 3) line-by-line profile of the vanilla call + its internals ---
%lprun -f dn2dem_pos -f demmap_fn -f dem_pix dn2dem_pos(frame, edn, T_RESP, T_RESP_LOGT, TEMPS, nmu=42)


In [12]:


def time_one(frame_6hw, T_RESP, T_RESP_LOGT, TEMPS, nmu=42):
    f = np.moveaxis(frame_6hw, 0, -1).astype(np.float64, copy=False)
    f = np.clip(np.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0), 0, None)
    e = np.sqrt(f) + 1e-6
    t0 = time.perf_counter()
    _ = dn2dem_pos(f, e, T_RESP, T_RESP_LOGT, TEMPS, nmu=nmu)
    return time.perf_counter() - t0

for sz in [14, 64, 256, 512]:  # scale up as feasible
    dt = time_one(STACK[0, :, :sz, :sz], T_RESP, T_RESP_LOGT, TEMPS)
    dems = sz*sz/dt
    print(f"{sz}x{sz}: {dt:.3f}s  |  {dems:,.0f} DEM/s")


14x14: 0.007s  |  26,644 DEM/s


100%|██████████| 40.0/40.0 [00:00<00:00, 63.5 x10^2 DEM/s]


64x64: 0.771s  |  5,313 DEM/s


100%|██████████| 655/655 [00:00<00:00, 995 x10^2 DEM/s]  


256x256: 0.849s  |  77,181 DEM/s


100%|██████████| 2.62k/2.62k [00:01<00:00, 2.05k x10^2 DEM/s]


512x512: 1.659s  |  157,990 DEM/s


In [13]:
# 15x15 = 225 px ⇒ triggers ProcessPool branch in demmap_pos
dt = time_one(STACK[0, :, :15, :15], T_RESP, T_RESP_LOGT, TEMPS)
print(f"15x15 (parallel branch): {dt:.3f}s  |  {(15*15)/dt:,.0f} DEM/s")


100%|██████████| 2.00/2.00 [00:00<00:00, 6.78 x10^2 DEM/s]

15x15 (parallel branch): 0.334s  |  673 DEM/s


In [ ]:
# baseline_profile.py
import os, sys, json, time, platform, statistics as stats
from pathlib import Path

# ----------------------------
# 0) Baseline environment caps
# ----------------------------
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np

# Project modules
from src.baseline.vendor.dn2dem_pos import dn2dem_pos
# demmap_pos exports demmap_pos() and dem_pix()
from demmap_pos import demmap_pos as demmap_fn, dem_pix

OUTDIR = Path("benchmark_out")
OUTDIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------
# 1) Data: reuse if present, else synthetic
# -----------------------------------------
def prepare_synthetic_responses(logT_min=5.5, logT_max=7.5, n_tresp=200, nt=24, nf=6):
    logT = np.linspace(logT_min, logT_max, n_tresp)  # T_RESP_LOGT
    centers = np.linspace(logT_min+0.2, logT_max-0.2, nf)
    width = 0.15
    T_RESP = np.exp(-0.5*((logT[:, None]-centers[None, :])/width)**2) + 1e-30
    TEMPS = np.logspace(logT_min, logT_max, nt+1)   # Kelvin edges
    return T_RESP, logT, TEMPS

def get_inputs_or_synthetic(globals_dict):
    if all(k in globals_dict for k in ("STACK", "T_RESP", "T_RESP_LOGT", "TEMPS")):
        STACK = globals_dict["STACK"]
        T_RESP = globals_dict["T_RESP"]
        T_RESP_LOGT = globals_dict["T_RESP_LOGT"]
        TEMPS = globals_dict["TEMPS"]
    else:
        # Minimal synthetic stack: 1 frame of 6×H×W; you can overwrite with your real STACK in the notebook.
        H, W = 1024, 1024
        nf = 6
        rng = np.random.default_rng(0)
        STACK = rng.random((1, nf, H, W), dtype=np.float32) * 1e3
        T_RESP, T_RESP_LOGT, TEMPS = prepare_synthetic_responses(n_tresp=200, nt=24, nf=nf)
    return STACK, T_RESP, T_RESP_LOGT, TEMPS

STACK, T_RESP, T_RESP_LOGT, TEMPS = get_inputs_or_synthetic(globals())

# choose dtype for compute
DTYPE = np.float32  # stick to one dtype across all runs for comparability

# ---------------------------------------
# 2) Wrapper: vanilla dn2dem_pos per tile
# ---------------------------------------
def run_dn2dem(frame_6hw, T_RESP, T_RESP_LOGT, TEMPS, nmu=42):
    """
    frame_6hw: (6, H, W)
    Returns (demmap, edemmap, logT_bins, chisq, dn_reg)
    """
    f = np.moveaxis(frame_6hw, 0, -1).astype(DTYPE, copy=False)  # (H,W,6)
    # sanitize + simple errors
    f = np.clip(np.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0), 0, None)
    e = np.sqrt(f, dtype=np.float32) + 1e-6
    # vanilla solver
    return dn2dem_pos(f, e, T_RESP, T_RESP_LOGT, TEMPS, nmu=nmu)

def time_one(frame_6hw, T_RESP, T_RESP_LOGT, TEMPS, nmu=42):
    t0 = time.perf_counter()
    _ = run_dn2dem(frame_6hw, T_RESP, T_RESP_LOGT, TEMPS, nmu=nmu)
    dt = time.perf_counter() - t0
    H, W = frame_6hw.shape[1], frame_6hw.shape[2]
    return dt, (H*W)/dt

# -------------------------------------
# 3) Wall-clock benchmark (CSV + .md)
# -------------------------------------
def benchmark_wallclock():
    sizes = [14, 64, 256, 1024]  # escalate carefully
    repeats = 5
    rows = []
    for sz in sizes:
        sz = min(sz, STACK.shape[2])  # clamp to available size
        frame = STACK[0, :, :sz, :sz]
        dts, thr = [], []
        # one warm-up
        _ = time_one(frame, T_RESP, T_RESP_LOGT, TEMPS)
        for _ in range(repeats):
            dt, dems = time_one(frame, T_RESP, T_RESP_LOGT, TEMPS)
            dts.append(dt); thr.append(dems)
        row = dict(
            size=f"{sz}x{sz}",
            H=sz, W=sz, nf=int(frame.shape[0]),
            repeats=repeats,
            time_mean=np.mean(dts), time_std=np.std(dts),
            time_median=stats.median(dts),
            dems_per_s_mean=np.mean(thr), dems_per_s_median=stats.median(thr),
            nmu=42, dtype=str(DTYPE)
        )
        rows.append(row)

    # Write CSV
    csv_path = OUTDIR / "baseline_wallclock.csv"
    import csv
    with csv_path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

    # Write a tiny Markdown summary
    md_path = OUTDIR / "summary.md"
    with md_path.open("w") as f:
        f.write("| Size | repeats | time_median [s] | DEMs/s (median) | dtype | nmu |\n")
        f.write("|------|---------|-----------------|------------------|-------|-----|\n")
        for r in rows:
            f.write(f"| {r['size']} | {r['repeats']} | {r['time_median']:.4f} | {r['dems_per_s_median']:,.0f} | {r['dtype']} | {r['nmu']} |\n")
    return rows

# ---------------------------------------
# 4) cProfile (save .prof + text summary)
# ---------------------------------------
def run_cprofile(sz=256):
    sz = min(sz, STACK.shape[2])
    frame = STACK[0, :, :sz, :sz]

    import cProfile, pstats, io
    pr = cProfile.Profile()
    pr.enable()
    _ = run_dn2dem(frame, T_RESP, T_RESP_LOGT, TEMPS, nmu=42)
    pr.disable()

    prof_path = OUTDIR / f"profile_dn2dem_pos_{sz}x{sz}.prof"
    pr.dump_stats(str(prof_path))

    txt_path = OUTDIR / f"profile_dn2dem_pos_{sz}x{sz}.txt"
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats("cumtime")
    ps.print_stats(40)
    with txt_path.open("w") as f:
        f.write(s.getvalue())
    return str(prof_path), str(txt_path)

# ------------------------------------------------
# 5) line_profiler (serial path, tiny crop; files)
# ------------------------------------------------
def run_line_profiler(sz=14):
    """
    Profiles dn2dem_pos (vanilla), demmap_pos (serial), dem_pix at 14x14 (na=196<200).
    Writes .lprof + text report. Skips gracefully if line_profiler not installed.
    """
    try:
        import line_profiler
    except Exception as e:
        with (OUTDIR / "line_profiler_skipped.txt").open("w") as f:
            f.write(f"line_profiler not available: {e}\n")
        return None, None

    sz = min(sz, STACK.shape[2])
    frame = STACK[0, :, :sz, :sz]

    lp = line_profiler.LineProfiler()
    lp.add_function(dn2dem_pos)
    lp.add_function(demmap_fn)
    lp.add_function(dem_pix)

    def _target():
        _ = run_dn2dem(frame, T_RESP, T_RESP_LOGT, TEMPS, nmu=42)

    lp_wrapper = lp(_target)
    lp_wrapper()  # run

    # Dump binary stats
    lprof_path = OUTDIR / f"lineprofile_{sz}x{sz}.lprof"
    lp.dump_stats(str(lprof_path))

    # Dump text
    txt_path = OUTDIR / f"lineprofile_{sz}x{sz}.txt"
    with txt_path.open("w") as f:
        lp.print_stats(stream=f)
    return str(lprof_path), str(txt_path)

# -----------------------------------
# 6) Env snapshot (for reproducibility)
# -----------------------------------
def write_env_snapshot():
    info = {
        "python": sys.version.replace("\n", " "),
        "executable": sys.executable,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "thread_caps": {
            "OMP_NUM_THREADS": os.environ.get("OMP_NUM_THREADS"),
            "OPENBLAS_NUM_THREADS": os.environ.get("OPENBLAS_NUM_THREADS"),
            "MKL_NUM_THREADS": os.environ.get("MKL_NUM_THREADS"),
            "VECLIB_MAXIMUM_THREADS": os.environ.get("VECLIB_MAXIMUM_THREADS"),
            "NUMEXPR_NUM_THREADS": os.environ.get("NUMEXPR_NUM_THREADS"),
        },
        "stack_shape": tuple(int(x) for x in STACK.shape),
        "tresp_shape": tuple(int(x) for x in T_RESP.shape),
        "tresp_logt_len": int(T_RESP_LOGT.shape[0]),
        "temps_len": int(TEMPS.shape[0]),
        "dtype": str(DTYPE),
    }
    (OUTDIR / "env.json").write_text(json.dumps(info, indent=2))

if __name__ == "__main__":
    write_env_snapshot()
    rows = benchmark_wallclock()
    prof_bin, prof_txt = run_cprofile(sz=256)               # adjust size if needed
    lbin, ltxt = run_line_profiler(sz=14)                   # 14x14 -> serial branch
    print("Baseline outputs written to:", OUTDIR.resolve())
